In [ ]:
#!/usr/bin/env python3
"""
Scherman Crypto Strategy - Condensed Version
A comprehensive crypto trading strategy with ML models, risk management, and live execution.
"""

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ccxt
from datetime import datetime, timedelta
import asyncio
import time
import json
from typing import Dict, List, Tuple, Optional
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import ta
import websocket
import threading
import queue
import logging
from concurrent.futures import ThreadPoolExecutor
import requests
import joblib
import torch
import torch.nn as nn


class CryptoDataManager:
    def __init__(self, config, exchange):
        self.config = config
        self.exchange = exchange
        
    def initialize(self):
        return True
        
    def get_historical_data(self, symbol, timeframe, days):
        try:
            since = int((datetime.now() - timedelta(days=days)).timestamp() * 1000)
            ohlcv = self.exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.set_index('timestamp', inplace=True)
            return df
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
            return pd.DataFrame()


class RiskManager:
    def __init__(self, config):
        self.config = config
        
    def validate_signal(self, symbol, signal, positions):
        if symbol in positions and positions[symbol]['size'] != 0:
            return False
        return signal['confidence'] > self.config.get('min_signal_confidence', 0.6)
        
    def calculate_position_size(self, symbol, signal, equity):
        risk_per_trade = self.config.get('risk_per_trade', 0.02)
        return equity * risk_per_trade


class PortfolioManager:
    def __init__(self, config, exchange):
        self.config = config
        self.exchange = exchange
        
    def get_total_equity(self):
        try:
            balance = self.exchange.fetch_balance()
            return balance['total']['USDT']
        except:
            return 10000  # Default for testing


class ExecutionEngine:
    def __init__(self, config, exchange):
        self.config = config
        self.exchange = exchange
        
    def place_order(self, symbol, side, size, order_type='market', reduce_only=False):
        try:
            if self.config.get('sandbox', True):
                # Simulate order execution in sandbox mode
                return {
                    'success': True,
                    'filled_size': size,
                    'average_price': 50000 + np.random.normal(0, 100),  # Mock price
                    'fees': size * 0.0005,
                    'order_id': f"sim_{int(time.time())}"
                }
            else:
                # Real order execution
                order = self.exchange.create_order(symbol, order_type, side, size)
                return {
                    'success': True,
                    'filled_size': order['filled'],
                    'average_price': order['average'],
                    'fees': order['fee']['cost'],
                    'order_id': order['id']
                }
        except Exception as e:
            return {'success': False, 'error': str(e)}


class PerformanceMonitor:
    def __init__(self, config):
        self.config = config
        
    def setup_performance_tracking(self):
        pass
        
    def setup_risk_monitoring(self):
        pass
        
    def setup_execution_monitoring(self):
        pass


class SchermanCryptoStrategy:
    def __init__(self, config: Dict):
        self.config = config
        self.okx_client = self._init_okx_client()
        
        # Core components
        self.data_manager = CryptoDataManager(config, self.okx_client)
        self.risk_manager = RiskManager(config)
        self.portfolio_manager = PortfolioManager(config, self.okx_client)
        self.execution_engine = ExecutionEngine(config, self.okx_client)
        self.monitor = PerformanceMonitor(config)
        
        # Trading state
        self.positions = {}
        self.trade_log = []
        self.equity_curve = []
        self.market_data = {}
        self.running = False
        
        # ML models and data
        self.ensemble_models = {}
        self.model_weights = {}
        self.historical_data = {}
        
        # Performance metrics
        self.realized_pnl = 0.0
        self.unrealized_pnl = 0.0
        self.portfolio_heat = 0.0
        self.strategy_performance = {}
        
        # Real-time data
        self.price_cache = {}
        self.technical_indicators = {}
        self.trade_queue = queue.Queue()
        self.data_queue = queue.Queue()
        self.thread_pool = ThreadPoolExecutor(max_workers=10)
        
    def _init_okx_client(self):
        return ccxt.okx({
            'apiKey': self.config['okx_api_key'],
            'secret': self.config['okx_secret'],
            'password': self.config['okx_passphrase'],
            'sandbox': self.config.get('sandbox', False),
            'enableRateLimit': True,
            'options': {'defaultType': 'swap'}
        })
        
    def initialize(self):
        print("🚀 Initializing Scherman Crypto Strategy...")
        try:
            self.data_manager.initialize()
            self.load_historical_data()
            self.initialize_models()
            self.monitor.setup_performance_tracking()
            self.start_background_tasks()
            print("✅ Strategy initialized successfully!")
            return True
        except Exception as e:
            print(f"❌ Initialization failed: {e}")
            return False
            
    def load_historical_data(self):
        print(f"📊 Loading historical data...")
        self.historical_data = {}
        for symbol in self.config['symbols']:
            try:
                data = self.data_manager.get_historical_data(
                    symbol, self.config['timeframe'], self.config['lookback_days']
                )
                if len(data) > 100:
                    self.historical_data[symbol] = data
                    print(f"✅ Loaded {len(data)} candles for {symbol}")
            except Exception as e:
                print(f"❌ Error loading data for {symbol}: {e}")
                
    def generate_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Generate comprehensive technical features"""
        features = pd.DataFrame(index=data.index)
        
        # Price features
        features['returns'] = data['close'].pct_change()
        features['log_returns'] = np.log(data['close'] / data['close'].shift(1))
        features['volatility'] = features['returns'].rolling(24).std()
        
        # Moving averages
        for period in [5, 10, 20, 50]:
            features[f'sma_{period}'] = data['close'].rolling(period).mean()
            features[f'ema_{period}'] = data['close'].ewm(span=period).mean()
            
        # Technical indicators
        features['rsi'] = ta.momentum.RSIIndicator(data['close'], window=14).rsi()
        
        macd = ta.trend.MACD(data['close'])
        features['macd'] = macd.macd()
        features['macd_signal'] = macd.macd_signal()
        features['macd_diff'] = macd.macd_diff()
        
        bb = ta.volatility.BollingerBands(data['close'], window=20)
        features['bb_upper'] = bb.bollinger_hband()
        features['bb_lower'] = bb.bollinger_lband()
        features['bb_middle'] = bb.bollinger_mavg()
        
        features['atr'] = ta.volatility.AverageTrueRange(
            data['high'], data['low'], data['close'], window=14
        ).average_true_range()
        
        features['adx'] = ta.trend.ADXIndicator(
            data['high'], data['low'], data['close'], window=14
        ).adx()
        
        # Volume features
        features['volume_sma'] = data['volume'].rolling(20).mean()
        features['volume_ratio'] = data['volume'] / features['volume_sma']
        
        # Price patterns
        features['high_low_ratio'] = data['high'] / data['low']
        features['close_position'] = (data['close'] - data['low']) / (data['high'] - data['low'])
        
        # Lag features
        for lag in [1, 2, 3, 6, 12, 24]:
            features[f'returns_lag_{lag}'] = features['returns'].shift(lag)
            features[f'volatility_lag_{lag}'] = features['volatility'].shift(lag)
            
        # Time features
        hour = pd.to_datetime(data.index).hour
        features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
        features['hour_cos'] = np.cos(2 * np.pi * hour / 24)
        
        # Clean and fill
        features = features.replace([np.inf, -np.inf], np.nan)
        features = features.fillna(method='ffill').fillna(0)
        
        return features
        
    def generate_labels(self, data: pd.DataFrame) -> pd.Series:
        """Generate prediction labels based on future returns"""
        forward_returns = data['close'].pct_change().shift(-self.config['prediction_horizon'])
        labels = pd.cut(
            forward_returns,
            bins=[-np.inf, -0.02, -0.005, 0.005, 0.02, np.inf],
            labels=[0, 1, 2, 3, 4]
        ).astype(int)
        return labels
        
    def initialize_models(self):
        print("🧠 Training ML models...")
        for symbol in self.historical_data.keys():
            try:
                data = self.historical_data[symbol]
                features = self.generate_features(data)
                labels = self.generate_labels(data)
                
                if len(features) > 500:
                    self.train_symbol_models(symbol, features, labels)
                    print(f"✅ Models trained for {symbol}")
            except Exception as e:
                print(f"❌ Error training models for {symbol}: {e}")
                
    def train_symbol_models(self, symbol: str, features: pd.DataFrame, labels: pd.Series):
        """Train ensemble of ML models for a symbol"""
        valid_idx = ~(features.isna().any(axis=1) | labels.isna())
        X = features[valid_idx]
        y = labels[valid_idx]
        
        if len(X) < 1000:
            return
            
        # Train/test split
        train_size = int(0.8 * len(X))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]
        
        # Scale features
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train models
        models = {
            'rf': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
            'xgb': xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42),
            'lgb': lgb.LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=-1)
        }
        
        trained_models = {}
        model_scores = {}
        
        for name, model in models.items():
            try:
                if name == 'rf':
                    model.fit(X_train, y_train)
                    score = model.score(X_test, y_test)
                else:
                    model.fit(X_train_scaled, y_train)
                    score = model.score(X_test_scaled, y_test)
                    
                trained_models[name] = model
                model_scores[name] = score
            except Exception as e:
                print(f"❌ Error training {name}: {e}")
                
        self.ensemble_models[symbol] = trained_models
        self.model_weights[symbol] = self._calculate_model_weights(model_scores)
        
        # Save models
        joblib.dump({
            'models': trained_models,
            'scaler': scaler,
            'weights': self.model_weights[symbol],
            'feature_names': X.columns.tolist()
        }, f'models_{symbol}.pkl')
        
    def _calculate_model_weights(self, scores: Dict) -> Dict:
        total_score = sum(scores.values())
        if total_score == 0:
            return {name: 1/len(scores) for name in scores}
        return {name: score/total_score for name, score in scores.items()}
        
    def start_background_tasks(self):
        self.running = True
        self.thread_pool.submit(self._signal_generator_loop)
        self.thread_pool.submit(self._execution_loop)
        self.thread_pool.submit(self._performance_tracker)
        
    def _signal_generator_loop(self):
        """Main signal generation loop"""
        while self.running:
            try:
                for symbol in self.config['symbols']:
                    if symbol in self.ensemble_models:
                        signal = self._generate_signal(symbol)
                        if signal:
                            self.trade_queue.put((symbol, signal))
                            
                time.sleep(self.config.get('signal_interval', 300))
            except Exception as e:
                print(f"❌ Signal generator error: {e}")
                time.sleep(60)
                
    def _generate_signal(self, symbol: str) -> Dict:
        """Generate trading signal using ML models"""
        try:
            # Get latest data
            data = self.data_manager.get_historical_data(symbol, self.config['timeframe'], 30)
            if len(data) < 100:
                return None
                
            features = self.generate_features(data)
            if features.empty:
                return None
                
            latest_features = features.iloc[-1:].fillna(0)
            
            # Get model predictions
            models = self.ensemble_models[symbol]
            weights = self.model_weights[symbol]
            ensemble_proba = np.zeros(5)
            
            for model_name, model in models.items():
                try:
                    if model_name == 'rf':
                        proba = model.predict_proba(latest_features)[0]
                    else:
                        scaler = joblib.load(f'models_{symbol}.pkl')['scaler']
                        scaled_features = scaler.transform(latest_features)
                        proba = model.predict_proba(scaled_features)[0]
                        
                    ensemble_proba += proba * weights.get(model_name, 0)
                except Exception as e:
                    continue
                    
            predicted_class = np.argmax(ensemble_proba)
            confidence = np.max(ensemble_proba)
            
            signal_mapping = {0: 'strong_sell', 1: 'sell', 2: 'hold', 3: 'buy', 4: 'strong_buy'}
            signal = signal_mapping[predicted_class]
            
            # Check confidence threshold
            if confidence < self.config.get('min_signal_confidence', 0.6):
                return None
                
            # Get current price
            current_price = data['close'].iloc[-1]
            atr = latest_features['atr'].iloc[0] if 'atr' in latest_features else current_price * 0.02
            
            # Generate signal
            if signal in ['strong_buy', 'buy']:
                return {
                    'symbol': symbol,
                    'signal': signal,
                    'direction': 'long',
                    'confidence': confidence,
                    'entry_price': current_price,
                    'stop_loss': current_price - (2 * atr),
                    'take_profit': current_price + (3 * atr),
                    'timestamp': datetime.now()
                }
            elif signal in ['strong_sell', 'sell']:
                return {
                    'symbol': symbol,
                    'signal': signal,
                    'direction': 'short',
                    'confidence': confidence,
                    'entry_price': current_price,
                    'stop_loss': current_price + (2 * atr),
                    'take_profit': current_price - (3 * atr),
                    'timestamp': datetime.now()
                }
                
            return None
        except Exception as e:
            print(f"❌ Signal generation error for {symbol}: {e}")
            return None
            
    def _execution_loop(self):
        """Main execution loop"""
        while self.running:
            try:
                if not self.trade_queue.empty():
                    symbol, signal = self.trade_queue.get()
                    self._execute_signal(symbol, signal)
                    
                self._manage_positions()
                time.sleep(1)
            except Exception as e:
                print(f"❌ Execution loop error: {e}")
                time.sleep(5)
                
    def _execute_signal(self, symbol: str, signal: Dict):
        """Execute trading signal"""
        try:
            if not self.risk_manager.validate_signal(symbol, signal, self.positions):
                return
                
            position_size = self.risk_manager.calculate_position_size(
                symbol, signal, self.portfolio_manager.get_total_equity()
            )
            
            if position_size == 0:
                return
                
            order_result = self.execution_engine.place_order(
                symbol=symbol,
                side=signal['direction'],
                size=position_size,
                order_type='market'
            )
            
            if order_result['success']:
                self._update_position(symbol, order_result, signal)
                self._log_trade(symbol, order_result, signal)
                print(f"✅ Order executed: {symbol} {signal['direction']} {position_size}")
            else:
                print(f"❌ Order failed: {symbol} - {order_result['error']}")
                
        except Exception as e:
            print(f"❌ Signal execution error: {e}")
            
    def _update_position(self, symbol: str, order_result: Dict, signal: Dict):
        """Update position after order execution"""
        if symbol not in self.positions:
            self.positions[symbol] = {
                'size': 0, 'side': None, 'entry_price': 0, 'stop_loss': None,
                'take_profit': None, 'timestamp': None
            }
            
        position = self.positions[symbol]
        position['size'] = order_result['filled_size']
        position['side'] = signal['direction']
        position['entry_price'] = order_result['average_price']
        position['stop_loss'] = signal['stop_loss']
        position['take_profit'] = signal['take_profit']
        position['timestamp'] = datetime.now()
        
    def _log_trade(self, symbol: str, order_result: Dict, signal: Dict):
        """Log trade details"""
        self.trade_log.append({
            'timestamp': datetime.now(),
            'symbol': symbol,
            'side': signal['direction'],
            'size': order_result['filled_size'],
            'price': order_result['average_price'],
            'confidence': signal['confidence'],
            'fees': order_result.get('fees', 0)
        })
        
    def _manage_positions(self):
        """Manage open positions - stop losses, take profits"""
        for symbol, position in self.positions.items():
            if position['size'] == 0:
                continue
                
            # Get current price
            current_data = self.data_manager.get_historical_data(symbol, '1m', 1)
            if current_data.empty:
                continue
                
            current_price = current_data['close'].iloc[-1]
            
            # Check stop loss
            should_close = False
            reason = ""
            
            if position['side'] == 'long' and current_price <= position['stop_loss']:
                should_close = True
                reason = "Stop Loss"
            elif position['side'] == 'short' and current_price >= position['stop_loss']:
                should_close = True
                reason = "Stop Loss"
            elif position['side'] == 'long' and current_price >= position['take_profit']:
                should_close = True
                reason = "Take Profit"
            elif position['side'] == 'short' and current_price <= position['take_profit']:
                should_close = True
                reason = "Take Profit"
                
            if should_close:
                self._close_position(symbol, current_price, reason)
                
    def _close_position(self, symbol: str, exit_price: float, reason: str):
        """Close position"""
        position = self.positions[symbol]
        if position['size'] == 0:
            return
            
        close_side = 'sell' if position['side'] == 'long' else 'buy'
        
        order_result = self.execution_engine.place_order(
            symbol=symbol,
            side=close_side,
            size=abs(position['size']),
            order_type='market',
            reduce_only=True
        )
        
        if order_result['success']:
            pnl = self._calculate_pnl(position, exit_price)
            self.realized_pnl += pnl
            print(f"🔄 Position closed: {symbol} {reason} PnL: ${pnl:.2f}")
            
            # Reset position
            self.positions[symbol] = {
                'size': 0, 'side': None, 'entry_price': 0, 'stop_loss': None,
                'take_profit': None, 'timestamp': None
            }
            
    def _calculate_pnl(self, position: Dict, exit_price: float) -> float:
        """Calculate position PnL"""
        if position['side'] == 'long':
            return (exit_price - position['entry_price']) * position['size']
        else:
            return (position['entry_price'] - exit_price) * position['size']
            
    def _performance_tracker(self):
        """Track and display performance metrics"""
        while self.running:
            try:
                self._update_performance_metrics()
                time.sleep(300)  # Update every 5 minutes
            except Exception as e:
                print(f"❌ Performance tracker error: {e}")
                time.sleep(600)
                
    def _update_performance_metrics(self):
        """Calculate and display performance"""
        total_equity = self.portfolio_manager.get_total_equity()
        
        # Calculate unrealized PnL
        self.unrealized_pnl = 0
        for symbol, position in self.positions.items():
            if position['size'] != 0:
                try:
                    current_data = self.data_manager.get_historical_data(symbol, '1m', 1)
                    if not current_data.empty:
                        current_price = current_data['close'].iloc[-1]
                        pnl = self._calculate_pnl(position, current_price)
                        self.unrealized_pnl += pnl
                except:
                    continue
                    
        # Update equity curve
        self.equity_curve.append({
            'timestamp': datetime.now(),
            'equity': total_equity + self.unrealized_pnl,
            'realized_pnl': self.realized_pnl,
            'unrealized_pnl': self.unrealized_pnl
        })
        
        # Display performance
        if len(self.equity_curve) > 1:
            print(f"\n📊 Performance: Equity: ${total_equity:.2f} | "
                  f"Realized: ${self.realized_pnl:.2f} | "
                  f"Unrealized: ${self.unrealized_pnl:.2f}")
                  
    def run_backtest(self, start_date: str = None, end_date: str = None):
        """Run strategy backtest"""
        print("📈 Running backtest...")
        
        all_trades = []
        for symbol in self.historical_data.keys():
            symbol_data = self.historical_data[symbol].copy()
            
            if start_date:
                symbol_data = symbol_data[symbol_data.index >= start_date]
            if end_date:
                symbol_data = symbol_data[symbol_data.index <= end_date]
                
            if len(symbol_data) < 500:
                continue
                
            features = self.generate_features(symbol_data)
            labels = self.generate_labels(symbol_data)
            
            # Train on first 70%, test on remaining 30%
            train_size = int(0.7 * len(features))
            self.train_symbol_models(symbol, features[:train_size], labels[:train_size])
            
            # Backtest on test period
            symbol_trades = self._backtest_symbol(symbol, features[train_size:], symbol_data[train_size:])
            all_trades.extend(symbol_trades)
            
        if all_trades:
            results = self._calculate_backtest_results(all_trades)
            self._display_backtest_results(results)
            return results
        return {}
        
    def _backtest_symbol(self, symbol: str, features: pd.DataFrame, price_data: pd.DataFrame):
        """Backtest single symbol"""
        trades = []
        position = None
        
        for i in range(len(features)):
            current_features = features.iloc[i:i+1]
            current_price = price_data.iloc[i]['close']
            
            signal = self._generate_backtest_signal(symbol, current_features)
            
            if signal and position is None:
                # Open position
                position = {
                    'symbol': symbol,
                    'entry_time': price_data.index[i],
                    'entry_price': current_price,
                    'direction': signal['direction'],
                    'stop_loss': signal['stop_loss'],
                    'take_profit': signal['take_profit']
                }
                
            elif position is not None:
                # Check exit conditions
                should_exit = False
                exit_reason = 'hold'
                
                if position['direction'] == 'long':
                    if current_price <= position['stop_loss']:
                        should_exit, exit_reason = True, 'stop_loss'
                    elif current_price >= position['take_profit']:
                        should_exit, exit_reason = True, 'take_profit'
                else:
                    if current_price >= position['stop_loss']:
                        should_exit, exit_reason = True, 'stop_loss'
                    elif current_price <= position['take_profit']:
                        should_exit, exit_reason = True, 'take_profit'
                        
                if should_exit:
                    pnl_pct = ((current_price / position['entry_price'] - 1) * 100 
                              if position['direction'] == 'long'
                              else (position['entry_price'] / current_price - 1) * 100)
                              
                    trades.append({
                        'symbol': symbol,
                        'entry_time': position['entry_time'],
                        'exit_time': price_data.index[i],
                        'entry_price': position['entry_price'],
                        'exit_price': current_price,
                        'direction': position['direction'],
                        'pnl_pct': pnl_pct,
                        'exit_reason': exit_reason
                    })
                    position = None
                    
        return trades
        
    def _generate_backtest_signal(self, symbol: str, features: pd.DataFrame):
        """Generate signal for backtesting"""
        if symbol not in self.ensemble_models or features.empty:
            return None
            
        try:
            latest_features = features.fillna(0)
            models = self.ensemble_models[symbol]
            weights = self.model_weights[symbol]
            ensemble_proba = np.zeros(5)
            
            for model_name, model in models.items():
                if model_name == 'rf':
                    proba = model.predict_proba(latest_features)[0]
                else:
                    scaler = joblib.load(f'models_{symbol}.pkl')['scaler']
                    scaled_features = scaler.transform(latest_features)
                    proba = model.predict_proba(scaled_features)[0]
                    
                ensemble_proba += proba * weights.get(model_name, 0)
                
            predicted_class = np.argmax(ensemble_proba)
            confidence = np.max(ensemble_proba)
            
            if confidence < 0.6:
                return None
                
            signal_mapping = {0: 'strong_sell', 1: 'sell', 2: 'hold', 3: 'buy', 4: 'strong_buy'}
            signal = signal_mapping[predicted_class]
            
            if signal in ['strong_buy', 'buy']:
                return {'direction': 'long', 'stop_loss': 0.95, 'take_profit': 1.06}
            elif signal in ['strong_sell', 'sell']:
                return {'direction': 'short', 'stop_loss': 1.05, 'take_profit': 0.94}
                
            return None
        except:
            return None
            
    def _calculate_backtest_results(self, trades: List[Dict]):
        """Calculate backtest performance metrics"""
        if not trades:
            return {}
            
        trades_df = pd.DataFrame(trades)
        
        total_trades = len(trades_df)
        winning_trades = len(trades_df[trades_df['pnl_pct'] > 0])
        win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
        
        total_return = trades_df['pnl_pct'].sum()
        avg_return = trades_df['pnl_pct'].mean()
        
        profits = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct'].sum()
        losses = abs(trades_df[trades_df['pnl_pct'] < 0]['pnl_pct'].sum())
        profit_factor = profits / losses if losses > 0 else 0
        
        equity_curve = (1 + trades_df['pnl_pct'] / 100).cumprod()
        peak = equity_curve.expanding().max()
        drawdown = (equity_curve - peak) / peak
        max_drawdown = abs(drawdown.min()) * 100
        
        returns = trades_df['pnl_pct'] / 100
        sharpe_ratio = (returns.mean() / returns.std() * np.sqrt(252) 
                       if returns.std() > 0 else 0)
        
        return {
            'total_return': total_return,
            'win_rate': win_rate,
            'total_trades': total_trades,
            'profit_factor': profit_factor,
            'max_drawdown': max_drawdown,
            'sharpe_ratio': sharpe_ratio,
            'avg_return_per_trade': avg_return,
            'equity_curve': equity_curve,
            'trades': trades_df
        }
        
    def _display_backtest_results(self, results: Dict):
        """Display backtest results"""
        print("\n" + "="*60)
        print("🏆 BACKTEST RESULTS - SCHERMAN CRYPTO STRATEGY")
        print("="*60)
        print(f"📊 Total Return: {results['total_return']:.2f}%")
        print(f"📊 Win Rate: {results['win_rate']:.1f}%")
        print(f"📊 Profit Factor: {results['profit_factor']:.2f}")
        print(f"📊 Max Drawdown: {results['max_drawdown']:.2f}%")
        print(f"📊 Sharpe Ratio: {results['sharpe_ratio']:.2f}")
        print(f"📊 Total Trades: {results['total_trades']}")
        print(f"📊 Avg Return/Trade: {results['avg_return_per_trade']:.2f}%")
        
    def run_live_trading(self):
        """Run live trading mode"""
        print("🔴 STARTING LIVE TRADING MODE")
        print("⚠️  WARNING: This will place real trades!")
        
        confirmation = input("Type 'YES' to confirm: ")
        if confirmation != 'YES':
            print("❌ Live trading cancelled")
            return
            
        print("🟢 Live trading confirmed - Starting execution...")
        
        try:
            while self.running:
                time.sleep(60)
        except KeyboardInterrupt:
            print("\n🛑 Shutting down...")
            self.running = False


# Configuration
config = {
    'okx_api_key': 'your_okx_api_key',
    'okx_secret': 'your_okx_secret',
    'okx_passphrase': 'your_okx_passphrase',
    'sandbox': True,
    'symbols': ['BTC-USDT-SWAP', 'ETH-USDT-SWAP', 'SOL-USDT-SWAP'],
    'timeframe': '1h',
    'lookback_days': 365,
    'prediction_horizon': 4,
    'min_signal_confidence': 0.65,
    'risk_per_trade': 0.02,
    'signal_interval': 300
}

# Main execution
if __name__ == "__main__":
    strategy = SchermanCryptoStrategy(config)
    
    if strategy.initialize():
        choice = input("\nSelect mode:\n1. Backtest\n2. Live Trading\nEnter choice (1/2): ")
        
        if choice == "1":
            results = strategy.run_backtest()
        elif choice == "2":
            strategy.run_live_trading()
        else:
            print("Invalid choice")
    else:
        print("Failed to initialize strategy")